In [ ]:
const taskDetailsChatEndRef = React.useRef<HTMLDivElement>(null);

// Scroll to bottom of Task Details Side Panel Chat
useEffect(() => {
  if (reviewTask && (mobileTab === 'chat' || rightPanelTab === 'chat')) {
    setTimeout(() => {
        taskDetailsChatEndRef.current?.scrollIntoView({ behavior: 'smooth' });
    }, 100);
  }
}, [reviewTask, mobileTab, rightPanelTab]);


In [ ]:
const [socket, setSocket] = useState<Socket | null>(null);
const [boardTypingUsers, setBoardTypingUsers] = useState<string[]>([]);
const [taskTypingUsers, setTaskTypingUsers] = useState<Record<string, string[]>>({});
const typingTimeoutRef = React.useRef<NodeJS.Timeout | null>(null);

// Initialize Socket Connection
useEffect(() => {
  if (!tableId) return;

  const newSocket = io(SERVER_URL);
  setSocket(newSocket);

  newSocket.on('connect', () => {
    console.log('Connected to socket server');
    newSocket.emit('join_board', tableId);
  });

  newSocket.on('typing_board', ({ user }) => {
    setBoardTypingUsers(prev => {
      if (!prev.includes(user)) return [...prev, user];
      return prev;
    });
  });

  newSocket.on('stop_typing_board', ({ user }) => {
    setBoardTypingUsers(prev => prev.filter(u => u !== user));
  });

  newSocket.on('typing_task', ({ taskId, user }) => {
    setTaskTypingUsers(prev => {
      const current = prev[taskId] || [];
      if (!current.includes(user)) {
        return { ...prev, [taskId]: [...current, user] };
      }
      return prev;
    });
  });

  newSocket.on('stop_typing_task', ({ taskId, user }) => {
    setTaskTypingUsers(prev => {
      const current = prev[taskId] || [];
      return { ...prev, [taskId]: current.filter(u => u !== user) };
    });
  });

  return () => {
    newSocket.disconnect();
  };
}, [tableId]);

const handleBoardTyping = () => {
  if (socket) {
    socket.emit('typing_board', { boardId: tableId, user: currentUser?.name || 'User' });
    
    if (typingTimeoutRef.current) clearTimeout(typingTimeoutRef.current);
    
    typingTimeoutRef.current = setTimeout(() => {
      socket.emit('stop_typing_board', { boardId: tableId, user: currentUser?.name || 'User' });
    }, 2000);
  }
};

const handleTaskTyping = (taskId: string) => {
  if (socket) {
    socket.emit('typing_task', { boardId: tableId, taskId, user: currentUser?.name || 'User' });
    
    // We need a separate timeout per task if we want robust per-task typing debounce,
    // but a global one often suffices if user focuses one input at a time.
    // However, if they type in two places rapidly, it might be weird.
    // For simplicity, let's reuse the timeout ref but maybe clear it only if it was for the same task? 
    // Or just clear it regardless, assuming single user focus.
    
    if (typingTimeoutRef.current) clearTimeout(typingTimeoutRef.current);
    
    typingTimeoutRef.current = setTimeout(() => {
      socket.emit('stop_typing_task', { boardId: tableId, taskId, user: currentUser?.name || 'User' });
    }, 2000);
  }
};


In [ ]:
print("Hello World")